# Sparse Autoencoders

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_Mech_Interp/blob/main/lectures/03_transformer_features/exercises/01_saes/notebook.ipynb)

In [ ]:
import os, importlib

if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_Mech_Interp/main/lectures/03_transformer_features/exercises/01_saes/utils.py",
        "utils.py"
    )
    %pip install transformer_lens einops jaxtyping typeguard plotly "numpy==2.0.2" "transformers>=4.38,<4.52" -q

import utils
importlib.reload(utils)
from utils import (
    test_standard_sae,
    test_topk_sae,
    test_batch_topk_sae,
    test_jumprelu_sae,
    test_train_sae,
    test_dead_neuron_tracker,
    score_sae,
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import einsum, rearrange
from jaxtyping import Float
from torch import Tensor
import plotly.graph_objects as go
from transformer_lens import HookedTransformer
from transformer_lens.utils import get_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gelu-2l", device=device)
d_model = model.cfg.d_model
print(f"Model: gelu-2l, d_model={d_model}, device={device}")

## Collecting Activations

We collect MLP output activations from layer 0 by running C4 text through the model.
These are the vectors our SAE will learn to reconstruct sparsely.

In [ ]:
dataset = get_dataset("c4")
all_acts = []
for i in range(200):
    tokens = model.to_tokens(dataset["text"][i], prepend_bos=True)[:, :128]
    _, cache = model.run_with_cache(tokens, names_filter="blocks.0.hook_mlp_out")
    all_acts.append(cache["blocks.0.hook_mlp_out"].detach().reshape(-1, d_model))
activations = torch.cat(all_acts).to(device)
print(f"Collected {activations.shape[0]} activation vectors of dimension {activations.shape[1]}")

## SAE Architecture Recap

An SAE maps an input $x \in \mathbb{R}^{d_{\text{model}}}$ to a sparse code $z \in \mathbb{R}^{d_{\text{dict}}}$ and back:

$$z = \text{activation}\big(W_{\text{enc}}\,(x - b_{\text{dec}}) + b_{\text{enc}}\big)$$
$$\hat{x} = W_{\text{dec}}\, z + b_{\text{dec}}$$

where $W_{\text{enc}} \in \mathbb{R}^{d_{\text{dict}} \times d_{\text{model}}}$, $W_{\text{dec}} \in \mathbb{R}^{d_{\text{model}} \times d_{\text{dict}}}$.

Note the **pre-encoder bias subtraction**: we subtract $b_{\text{dec}}$ before encoding. This lets the decoder bias capture the mean of the data, so the encoder sees zero-centered inputs.

The decoder columns are constrained to **unit norm** after each gradient step. This prevents the model from making decoder vectors large and encoder outputs small (or vice versa).

Different SAE variants differ only in their **activation function** and **sparsity term**.

## Exercise 1: Standard SAE

The standard SAE uses ReLU activation and an L1 sparsity penalty:

$$\mathcal{L} = \|x - \hat{x}\|_2^2 + \lambda \cdot \|z\|_1$$

Implement the class below. Use `einsum` from einops for all matrix multiplications.

**Einsum syntax reminder**: `einsum(A, B, "i j, ... j -> ... i")` computes $A \cdot B^T$ along the last dim of B.

In [ ]:
class StandardSAE(nn.Module):
    def __init__(self, d_model: int, d_dict: int):
        super().__init__()
        self.d_model = d_model
        self.d_dict = d_dict
        # TODO: Initialize W_enc (d_dict, d_model), b_enc (d_dict),
        #       W_dec (d_model, d_dict), b_dec (d_model)
        # Use nn.init.kaiming_uniform_ for weights, zeros for biases
        # Then normalize W_dec columns to unit norm
        pass

    def encode(self, x: Float[Tensor, "... d_model"]) -> Float[Tensor, "... d_dict"]:
        """ReLU(W_enc @ (x - b_dec) + b_enc)"""
        # TODO (~1 line)
        pass

    def decode(self, z: Float[Tensor, "... d_dict"]) -> Float[Tensor, "... d_model"]:
        """W_dec @ z + b_dec"""
        # TODO (~1 line)
        pass

    @torch.no_grad()
    def normalize_decoder(self):
        """Normalize W_dec columns to unit norm."""
        # TODO (~1 line)
        pass

    def forward(self, x: Float[Tensor, "batch d_model"], l1_coeff: float = 1.0):
        """Returns (x_hat, z, loss) where loss = MSE + l1_coeff * L1."""
        # TODO (~4 lines)
        pass

In [ ]:
test_standard_sae(StandardSAE, activations, device)

<details>
<summary><b>Hint</b></summary>

For encode: `F.relu(einsum(self.W_enc, x - self.b_dec, "d_dict d_model, ... d_model -> ... d_dict") + self.b_enc)`

For normalize: `F.normalize(self.W_dec.data, dim=0)` normalizes along the first dimension (each column).
</details>

<details>
<summary><b>Solution</b></summary>

```python
class StandardSAE(nn.Module):
    def __init__(self, d_model: int, d_dict: int):
        super().__init__()
        self.d_model = d_model
        self.d_dict = d_dict
        self.W_enc = nn.Parameter(torch.empty(d_dict, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_dict))
        self.W_dec = nn.Parameter(torch.empty(d_model, d_dict))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

    def encode(self, x: Float[Tensor, "... d_model"]) -> Float[Tensor, "... d_dict"]:
        return F.relu(einsum(self.W_enc, x - self.b_dec, "d_dict d_model, ... d_model -> ... d_dict") + self.b_enc)

    def decode(self, z: Float[Tensor, "... d_dict"]) -> Float[Tensor, "... d_model"]:
        return einsum(self.W_dec, z, "d_model d_dict, ... d_dict -> ... d_model") + self.b_dec

    @torch.no_grad()
    def normalize_decoder(self):
        self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

    def forward(self, x: Float[Tensor, "batch d_model"], l1_coeff: float = 1.0):
        z = self.encode(x)
        x_hat = self.decode(z)
        mse = (x - x_hat).pow(2).mean()
        l1 = z.abs().mean()
        return x_hat, z, mse + l1_coeff * l1
```
</details>

## Exercise 2: Top-K SAE

Instead of ReLU + L1, a Top-K SAE enforces sparsity structurally: only the top $k$ pre-activations are kept, the rest are zeroed out. No sparsity penalty is needed.

$$z_i = \begin{cases} \text{ReLU}(\text{pre}_i) & \text{if } \text{pre}_i \in \text{top-}k \\ 0 & \text{otherwise} \end{cases}$$

Use `torch.topk` to find the top-k values and indices, then `scatter_` to place them back.

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_model: int, d_dict: int, k: int):
        super().__init__()
        self.k = k
        # TODO: same parameters as StandardSAE
        pass

    def encode(self, x: Float[Tensor, "... d_model"]) -> Float[Tensor, "... d_dict"]:
        """Top-k selection on pre-activations."""
        # TODO (~4 lines):
        # 1. Compute pre-activations (same as StandardSAE but no ReLU)
        # 2. torch.topk to get top-k values and indices
        # 3. Create zeros, scatter ReLU(top-k values) back in
        pass

    def decode(self, z: Float[Tensor, "... d_dict"]) -> Float[Tensor, "... d_model"]:
        # TODO (~1 line)
        pass

    @torch.no_grad()
    def normalize_decoder(self):
        # TODO (~1 line)
        pass

    def forward(self, x: Float[Tensor, "batch d_model"]):
        """Returns (x_hat, z, loss). Loss is MSE only."""
        # TODO (~3 lines)
        pass

In [ ]:
test_topk_sae(TopKSAE, activations, device)

<details>
<summary><b>Solution</b></summary>

```python
class TopKSAE(nn.Module):
    def __init__(self, d_model: int, d_dict: int, k: int):
        super().__init__()
        self.k = k
        self.W_enc = nn.Parameter(torch.empty(d_dict, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_dict))
        self.W_dec = nn.Parameter(torch.empty(d_model, d_dict))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

    def encode(self, x: Float[Tensor, "... d_model"]) -> Float[Tensor, "... d_dict"]:
        pre = einsum(self.W_enc, x - self.b_dec, "d_dict d_model, ... d_model -> ... d_dict") + self.b_enc
        topk_vals, topk_idx = torch.topk(pre, self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topk_idx, F.relu(topk_vals))
        return z

    def decode(self, z: Float[Tensor, "... d_dict"]) -> Float[Tensor, "... d_model"]:
        return einsum(self.W_dec, z, "d_model d_dict, ... d_dict -> ... d_model") + self.b_dec

    @torch.no_grad()
    def normalize_decoder(self):
        self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

    def forward(self, x: Float[Tensor, "batch d_model"]):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z, (x - x_hat).pow(2).mean()
```
</details>

## Exercise 3: Batch Top-K SAE

Batch Top-K is a variant where the top-k selection happens across the **entire batch** rather than per sample. The total sparsity budget is $k \times \text{batch\_size}$, but individual samples can use more or fewer features.

This lets the SAE allocate more features to "harder" inputs that need them.

Implementation: flatten pre-activations to shape `(batch * d_dict)`, apply top-k with budget `k * batch_size`, then reshape back using `rearrange`.

In [ ]:
class BatchTopKSAE(nn.Module):
    def __init__(self, d_model: int, d_dict: int, k: int):
        super().__init__()
        self.k = k  # k per sample on average; total budget = k * batch_size
        # TODO: same parameters as StandardSAE
        pass

    def encode(self, x: Float[Tensor, "batch d_model"]) -> Float[Tensor, "batch d_dict"]:
        """Batch-level top-k selection."""
        # TODO (~6 lines):
        # 1. Compute pre-activations (batch, d_dict)
        # 2. rearrange to flat: "batch d_dict -> (batch d_dict)"
        # 3. torch.topk with budget = self.k * batch_size
        # 4. Create zeros, scatter ReLU values back
        # 5. rearrange back: "(batch d_dict) -> batch d_dict"
        pass

    def decode(self, z: Float[Tensor, "... d_dict"]) -> Float[Tensor, "... d_model"]:
        # TODO (~1 line)
        pass

    @torch.no_grad()
    def normalize_decoder(self):
        # TODO (~1 line)
        pass

    def forward(self, x: Float[Tensor, "batch d_model"]):
        """Returns (x_hat, z, loss). Loss is MSE only."""
        # TODO (~3 lines)
        pass

In [ ]:
test_batch_topk_sae(BatchTopKSAE, activations, device)

<details>
<summary><b>Solution</b></summary>

```python
class BatchTopKSAE(nn.Module):
    def __init__(self, d_model: int, d_dict: int, k: int):
        super().__init__()
        self.k = k
        self.W_enc = nn.Parameter(torch.empty(d_dict, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_dict))
        self.W_dec = nn.Parameter(torch.empty(d_model, d_dict))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

    def encode(self, x: Float[Tensor, "batch d_model"]) -> Float[Tensor, "batch d_dict"]:
        batch_size = x.shape[0]
        pre = einsum(self.W_enc, x - self.b_dec, "d_dict d_model, batch d_model -> batch d_dict") + self.b_enc
        flat = rearrange(pre, "batch d_dict -> (batch d_dict)")
        topk_vals, topk_idx = torch.topk(flat, self.k * batch_size)
        z_flat = torch.zeros_like(flat)
        z_flat.scatter_(0, topk_idx, F.relu(topk_vals))
        return rearrange(z_flat, "(batch d_dict) -> batch d_dict", batch=batch_size)

    def decode(self, z: Float[Tensor, "... d_dict"]) -> Float[Tensor, "... d_model"]:
        return einsum(self.W_dec, z, "d_model d_dict, ... d_dict -> ... d_model") + self.b_dec

    @torch.no_grad()
    def normalize_decoder(self):
        self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

    def forward(self, x: Float[Tensor, "batch d_model"]):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z, (x - x_hat).pow(2).mean()
```
</details>

## Exercise 4: JumpReLU SAE

JumpReLU uses a **learnable threshold** $\theta_i$ per feature. A feature fires only if its pre-activation exceeds the threshold, but the output is the full pre-activation value (not clipped):

$$z_i = \text{pre}_i \cdot \mathbb{1}[\text{pre}_i > \theta_i]$$

The indicator $\mathbb{1}[\cdot]$ has no gradient w.r.t. $\theta$, so we use a smooth proxy for the sparsity loss:

$$\mathcal{L}_0 = \frac{1}{B} \sum_b \sum_i \sigma\big(c \cdot (\text{pre}_{b,i} - \theta_i)\big)$$

where $\sigma$ is the sigmoid and $c$ (`ste_scale`) is a large constant that makes the sigmoid approximate the step function. The total loss is $\text{MSE} + \lambda_{L_0} \cdot \mathcal{L}_0$.

We store `log_threshold` as the parameter and compute $\theta = \exp(\text{log\_threshold})$ to keep thresholds positive.

In [ ]:
class JumpReLUSAE(nn.Module):
    def __init__(self, d_model: int, d_dict: int, threshold_init: float = 0.01):
        super().__init__()
        self.d_model = d_model
        self.d_dict = d_dict
        # TODO: W_enc, b_enc, W_dec, b_dec as before
        # TODO: log_threshold parameter (d_dict,) initialized to log(threshold_init)
        pass

    @property
    def threshold(self) -> Float[Tensor, "d_dict"]:
        return self.log_threshold.exp()

    def encode(self, x: Float[Tensor, "... d_model"]) -> Float[Tensor, "... d_dict"]:
        """Hard threshold with pass-through magnitudes."""
        # TODO (~3 lines):
        # 1. pre = einsum(W_enc, x - b_dec, ...) + b_enc
        # 2. mask = (pre > threshold).float()   [no gradient through this]
        # 3. return pre * mask
        pass

    def decode(self, z: Float[Tensor, "... d_dict"]) -> Float[Tensor, "... d_model"]:
        # TODO (~1 line)
        pass

    @torch.no_grad()
    def normalize_decoder(self):
        # TODO (~1 line)
        pass

    def forward(self, x: Float[Tensor, "batch d_model"], l0_coeff: float = 1e-3,
                ste_scale: float = 50.0):
        """
        Returns (x_hat, z, loss).
        loss = MSE + l0_coeff * L0_proxy
        L0_proxy = sigmoid(ste_scale * (pre - threshold)).sum(dim=-1).mean()
        """
        # TODO (~6 lines):
        # 1. Compute pre-activations
        # 2. z = pre * (pre > threshold).float()
        # 3. x_hat = decode(z)
        # 4. mse = (x - x_hat).pow(2).mean()
        # 5. l0_proxy = sigmoid(ste_scale * (pre - threshold)).sum(-1).mean()
        # 6. return x_hat, z, mse + l0_coeff * l0_proxy
        pass

In [ ]:
test_jumprelu_sae(JumpReLUSAE, activations, device)

<details>
<summary><b>Solution</b></summary>

```python
class JumpReLUSAE(nn.Module):
    def __init__(self, d_model: int, d_dict: int, threshold_init: float = 0.01):
        super().__init__()
        self.d_model = d_model
        self.d_dict = d_dict
        self.W_enc = nn.Parameter(torch.empty(d_dict, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_dict))
        self.W_dec = nn.Parameter(torch.empty(d_model, d_dict))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        self.log_threshold = nn.Parameter(torch.full((d_dict,), torch.tensor(threshold_init).log().item()))
        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

    @property
    def threshold(self) -> Float[Tensor, "d_dict"]:
        return self.log_threshold.exp()

    def encode(self, x: Float[Tensor, "... d_model"]) -> Float[Tensor, "... d_dict"]:
        pre = einsum(self.W_enc, x - self.b_dec, "d_dict d_model, ... d_model -> ... d_dict") + self.b_enc
        mask = (pre > self.threshold).float()
        return pre * mask

    def decode(self, z: Float[Tensor, "... d_dict"]) -> Float[Tensor, "... d_model"]:
        return einsum(self.W_dec, z, "d_model d_dict, ... d_dict -> ... d_model") + self.b_dec

    @torch.no_grad()
    def normalize_decoder(self):
        self.W_dec.data = F.normalize(self.W_dec.data, dim=0)

    def forward(self, x: Float[Tensor, "batch d_model"], l0_coeff: float = 1e-3,
                ste_scale: float = 50.0):
        pre = einsum(self.W_enc, x - self.b_dec, "d_dict d_model, ... d_model -> ... d_dict") + self.b_enc
        z = pre * (pre > self.threshold).float()
        x_hat = self.decode(z)
        mse = (x - x_hat).pow(2).mean()
        l0_proxy = torch.sigmoid(ste_scale * (pre - self.threshold)).sum(dim=-1).mean()
        return x_hat, z, mse + l0_coeff * l0_proxy
```
</details>

## Exercise 5: Training Loop

Write a training function that works with any of the SAE variants above.

Each step: sample a random batch, forward pass, backward, optimizer step, then normalize decoder weights.

In [ ]:
def train_sae(
    sae: nn.Module,
    activations: Float[Tensor, "n d_model"],
    n_steps: int = 1000,
    batch_size: int = 256,
    lr: float = 3e-4,
    log_every: int = 200,
    **forward_kwargs,
) -> list[float]:
    """
    Train an SAE on pre-collected activations.
    Returns list of loss values (one per step).
    Extra kwargs are passed to sae.forward (e.g. l1_coeff, l0_coeff).
    """
    # TODO (~12 lines):
    # 1. Move sae to same device as activations
    # 2. Create Adam optimizer
    # 3. Loop n_steps:
    #    a. Sample random batch indices
    #    b. Forward pass -> (x_hat, z, loss)
    #    c. zero_grad, backward, step
    #    d. normalize_decoder
    #    e. Append loss.item() to list
    #    f. Print every log_every steps
    pass

In [ ]:
test_train_sae(train_sae, StandardSAE, activations, device)

<details>
<summary><b>Solution</b></summary>

```python
def train_sae(
    sae: nn.Module,
    activations: Float[Tensor, "n d_model"],
    n_steps: int = 1000,
    batch_size: int = 256,
    lr: float = 3e-4,
    log_every: int = 200,
    **forward_kwargs,
) -> list[float]:
    sae = sae.to(activations.device)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    losses = []
    for step in range(n_steps):
        idx = torch.randint(0, len(activations), (batch_size,))
        x_hat, z, loss = sae(activations[idx], **forward_kwargs)
        opt.zero_grad()
        loss.backward()
        opt.step()
        sae.normalize_decoder()
        losses.append(loss.item())
        if step % log_every == 0:
            l0 = (z > 0).float().sum(dim=-1).mean().item()
            print(f"step {step:4d}  loss={loss.item():.4f}  L0={l0:.1f}")
    return losses
```
</details>

## Train and Compare All Variants

Let's train all four SAE types and compare their reconstruction quality vs. sparsity.

In [ ]:
d_dict = 4 * d_model  # 2048
k = 32

saes = {
    "Standard (L1)": StandardSAE(d_model, d_dict),
    "Top-K": TopKSAE(d_model, d_dict, k=k),
    "Batch Top-K": BatchTopKSAE(d_model, d_dict, k=k),
    "JumpReLU": JumpReLUSAE(d_model, d_dict),
}

forward_kwargs = {
    "Standard (L1)": {"l1_coeff": 1.0},
    "Top-K": {},
    "Batch Top-K": {},
    "JumpReLU": {"l0_coeff": 1e-3},
}

all_losses = {}
for name, sae in saes.items():
    print(f"\n--- {name} ---")
    losses = train_sae(sae, activations, n_steps=1000, **forward_kwargs[name])
    all_losses[name] = losses

In [ ]:
fig = go.Figure()
for name, losses in all_losses.items():
    fig.add_trace(go.Scatter(y=losses, name=name, mode="lines"))
fig.update_layout(title="SAE Training Loss", xaxis_title="Step", yaxis_title="Loss", yaxis_type="log")
fig.show()

In [ ]:
print(f"{'Variant':20s}  {'L0':>6s}  {'FVU':>8s}  {'Dead %':>7s}")
print("-" * 50)
with torch.no_grad():
    test_batch = activations[:1024]
    for name, sae in saes.items():
        x_hat, z, _ = sae(test_batch, **forward_kwargs[name])
        l0 = (z > 0).float().sum(dim=-1).mean().item()
        mse = (test_batch - x_hat).pow(2).mean().item()
        fvu = mse / test_batch.var().item()
        dead = ((z > 0).any(dim=0) == False).float().mean().item()
        print(f"{name:20s}  {l0:6.1f}  {fvu:8.4f}  {dead:6.1%}")

<details>
<summary><b>What to observe</b></summary>

- **Standard SAE**: Low FVU but high L0 (hundreds of active features). The L1 penalty fights against reconstruction quality.
- **Top-K / Batch Top-K**: Exactly k=32 active features. Clean sparsity, but 30-50% of dictionary neurons are dead.
- **JumpReLU**: Learns its own sparsity level. Threshold adapts during training.
- **Dead neurons**: A large fraction of neurons never fire. This is the main challenge in SAE training!
</details>

## Exercise 6: Dead Neuron Tracker

A neuron is "dead" if it never fires. In practice we track this with an exponential moving average (EMA) of firing rates.

Implement a tracker that:
- Maintains an EMA of per-neuron firing frequency
- Reports which neurons are dead (EMA below a threshold)

In [ ]:
class DeadNeuronTracker:
    def __init__(self, d_dict: int, threshold: float = 0.01, decay: float = 0.99):
        """
        Args:
            d_dict: number of dictionary features
            threshold: EMA above this = alive
            decay: EMA decay factor (0.99 = long memory)
        """
        self.threshold = threshold
        self.decay = decay
        # TODO: initialize EMA of firing rates (d_dict,) to 0.0 (assume dead until proven alive)
        pass

    def update(self, z: Float[Tensor, "batch d_dict"]):
        """Update EMA with firing rates from this batch."""
        # TODO (~2 lines):
        # 1. Compute fraction of batch where each neuron fires: (z > 0).float().mean(dim=0)
        # 2. Update EMA: ema = decay * ema + (1 - decay) * batch_rate
        pass

    @property
    def dead_mask(self) -> Float[Tensor, "d_dict"]:
        """Boolean mask: True = dead neuron."""
        # TODO (~1 line): dead if ema < threshold
        pass

In [ ]:
test_dead_neuron_tracker(DeadNeuronTracker, d_dict=2048)

<details>
<summary><b>Solution</b></summary>

```python
class DeadNeuronTracker:
    def __init__(self, d_dict: int, threshold: float = 0.01, decay: float = 0.99):
        self.threshold = threshold
        self.decay = decay
        self.ema = torch.zeros(d_dict)

    def update(self, z: Float[Tensor, "batch d_dict"]):
        batch_rate = (z > 0).float().mean(dim=0)
        self.ema = self.decay * self.ema + (1 - self.decay) * batch_rate

    @property
    def dead_mask(self) -> Float[Tensor, "d_dict"]:
        return self.ema < self.threshold
```
</details>

Let's use the tracker to visualize how dead neurons evolve during training:

In [ ]:
sae = TopKSAE(d_model, d_dict, k=k).to(device)
tracker = DeadNeuronTracker(d_dict)
opt = torch.optim.Adam(sae.parameters(), lr=3e-4)
dead_counts = []

for step in range(1000):
    idx = torch.randint(0, len(activations), (256,))
    x_hat, z, loss = sae(activations[idx])
    opt.zero_grad()
    loss.backward()
    opt.step()
    sae.normalize_decoder()
    tracker.update(z.detach())
    dead_counts.append(tracker.dead_mask.sum().item())

fig = go.Figure()
fig.add_trace(go.Scatter(y=dead_counts, mode="lines"))
fig.update_layout(title=f"Dead Neurons During Top-K Training (d_dict={d_dict}, k={k})",
                  xaxis_title="Step", yaxis_title="Dead neuron count")
fig.show()
print(f"Final: {dead_counts[-1]}/{d_dict} dead ({dead_counts[-1]/d_dict:.1%})")

## Exercise 7: Competition - Train the Best SAE!

**Goal**: Minimize FVU (fraction of variance unexplained) with L0 $\leq$ 50.

Use `score_sae` to evaluate. Lower score = better. L0 over budget incurs a steep penalty.

Strategies to try (mix and match!):

- **Auxiliary loss for dead neurons**: Reconstruct the residual $x - \hat{x}$ using only dead neurons' activations. This gives dead neurons gradient signal to become useful.
- **Learning rate schedule**: Warmup + cosine decay via `torch.optim.lr_scheduler`
- **Expansion ratio**: Try `d_dict = 8 * d_model` or `16 * d_model`
- **Tuning k**: What happens with k=16 vs k=64 vs k=128?
- **PCA initialization**: Initialize W_dec from PCA of activations
- **Neuron resampling**: Reinitialize dead neurons towards high-loss inputs (tricky to stabilize!)
- **Gradient clipping**: `torch.nn.utils.clip_grad_norm_`
- **Longer training**: More steps or larger batches
- **Different optimizer**: Try Adam with weight decay, or adjust beta1/beta2

There is no single right answer. Play around and see what works!

In [ ]:
# Example: score one of our trained SAEs
print("Baseline Top-K:")
score_sae(saes["Top-K"], activations, l0_budget=50)

In [ ]:
# Your improved SAE here!
# Try different architectures, hyperparameters, and training tricks.

# Example skeleton:
# my_sae = TopKSAE(d_model, d_dict=8*d_model, k=48).to(device)
# ... custom training loop with your improvements ...
# score_sae(my_sae, activations, l0_budget=50)